In [ ]:
import os, sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"
os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import logging
import utils.logger as _log_mod
def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers: return logger
    logger.setLevel(logging.DEBUG)
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S"))
    logger.addHandler(h)
    return logger
_log_mod.setup_logger = setup_logger

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CWD: {os.getcwd()}")

In [ ]:
!pip install lightning --quiet

In [ ]:
"""TFT v2 Walk-Forward 검증
3개월 단위 슬라이딩 윈도우로 시간 안정성 검증.
각 윈도우: 과거 데이터로 학습 -> 다음 3개월 예측 -> DA/AUC 측정
"""
import warnings, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import EarlyStopping

warnings.filterwarnings("ignore")
print(f"PyTorch: {torch.__version__}, Lightning: {pl.__version__}")

In [ ]:
# ===== 설정 =====
BASE_PATH = Path(os.environ.get("BASE_PATH"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
MODEL_SAVE_PATH = BASE_PATH / "models" / "tft_v2_wf"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

# Walk-Forward 설정
WF_START = "2021-01-01"      # 첫 번째 테스트 윈도우 시작
WF_END = "2026-01-31"        # 마지막 테스트 윈도우 끝
WF_STEP_MONTHS = 3            # 3개월 단위 슬라이딩
MAX_ENCODER_LENGTH = 30
BATCH_SIZE = 256
MAX_EPOCHS = 30               # WF는 윈도우 많으므로 에폭 줄임
LEARNING_RATE = 5e-4
HIDDEN_SIZE = 128
ATTENTION_HEADS = 4
DROPOUT = 0.2

CLEANED_FEATURES = ["vix_change", "vix", "macd_norm", "momentum_20d",
                    "relative_return", "high_low_range", "kospi200_return", "volume_ratio_5d"]

# Walk-Forward 윈도우 생성
windows = []
current = pd.Timestamp(WF_START)
end = pd.Timestamp(WF_END)
while current < end:
    test_end = current + pd.DateOffset(months=WF_STEP_MONTHS) - pd.Timedelta(days=1)
    if test_end > end: test_end = end
    train_end = current - pd.Timedelta(days=1)
    windows.append((str(train_end.date()), str(current.date()), str(test_end.date())))
    current += pd.DateOffset(months=WF_STEP_MONTHS)

print(f"Walk-Forward 윈도우: {len(windows)}개")
for i, (te, ts, tend) in enumerate(windows):
    print(f"  [{i:2d}] train_end={te}, test={ts} ~ {tend}")

In [ ]:
# ===== 데이터 로드 =====
df = pd.read_parquet(str(TFT_FEATURE_PATH))
df["date"] = pd.to_datetime(df["date"])
df["target_5d"] = df["target_5d"].astype(int)
CLEANED_FEATURES = [c for c in CLEANED_FEATURES if c in df.columns]
N_FEATURES = len(CLEANED_FEATURES)
print(f"데이터: {len(df):,} rows, {N_FEATURES} features")

In [ ]:
# ===== TFT v2 모델 정의 (동일) =====
class GatedLinearUnit(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc = nn.Linear(d, d); self.gate = nn.Linear(d, d)
    def forward(self, x): return self.fc(x) * torch.sigmoid(self.gate(x))

class GatedResidualNetwork(nn.Module):
    def __init__(self, d_in, d_h, d_out, drop=0.1):
        super().__init__()
        self.fc1=nn.Linear(d_in,d_h); self.fc2=nn.Linear(d_h,d_out)
        self.gate=GatedLinearUnit(d_out); self.norm=nn.LayerNorm(d_out)
        self.drop=nn.Dropout(drop); self.skip=nn.Linear(d_in,d_out) if d_in!=d_out else nn.Identity()
    def forward(self, x):
        r=self.skip(x); h=self.drop(F.elu(self.fc2(F.elu(self.fc1(x)))))
        return self.norm(self.gate(h)+r)

class VSN(nn.Module):
    def __init__(self, n_vars, d, drop=0.1):
        super().__init__()
        self.grns=nn.ModuleList([GatedResidualNetwork(d,d,d,drop) for _ in range(n_vars)])
        self.sg=GatedResidualNetwork(n_vars*d,d,n_vars,drop)
    def forward(self, x):
        B,S,V,D=x.shape
        p=torch.stack([self.grns[i](x[:,:,i,:]) for i in range(V)],dim=2)
        w=torch.softmax(self.sg(x.reshape(B,S,V*D)),dim=-1).unsqueeze(-1)
        return (p*w).sum(dim=2)

class TFTv2(pl.LightningModule):
    def __init__(self, n_feat, seq_len=30, d=128, heads=4, n_lstm=1, drop=0.2, n_cls=2, lr=5e-4, cw=None):
        super().__init__()
        self.save_hyperparameters(ignore=["cw"]); self.lr=lr; self.n_feat=n_feat
        self.fe=nn.Linear(1,d); self.vsn=VSN(n_feat,d,drop)
        self.lstm=nn.LSTM(d,d,n_lstm,batch_first=True)
        self.attn=nn.MultiheadAttention(d,heads,dropout=drop,batch_first=True)
        self.an=nn.LayerNorm(d); self.ag=GatedLinearUnit(d)
        self.go=GatedResidualNetwork(d,d,d,drop); self.head=nn.Linear(d,n_cls)
        self.loss_fn=nn.CrossEntropyLoss(weight=cw) if cw is not None else nn.CrossEntropyLoss()
    def forward(self, x):
        B,S,F=x.shape; x=self.fe(x.unsqueeze(-1)); x=self.vsn(x)
        x,_=self.lstm(x); a,_=self.attn(x,x,x); x=self.an(x+self.ag(a))
        return self.head(self.go(x[:,-1,:]))
    def training_step(self,b,_): x,y=b; l=self.loss_fn(self(x),y); self.log("train_loss",l,prog_bar=True); return l
    def validation_step(self,b,_):
        x,y=b; lo=self(x); self.log("val_loss",self.loss_fn(lo,y),prog_bar=True)
        self.log("val_acc",(torch.argmax(lo,1)==y).float().mean(),prog_bar=True)
    def configure_optimizers(self):
        o=torch.optim.AdamW(self.parameters(),lr=self.lr,weight_decay=1e-4)
        s=torch.optim.lr_scheduler.ReduceLROnPlateau(o,"min",0.5,patience=5,min_lr=1e-6)
        return {"optimizer":o,"lr_scheduler":{"scheduler":s,"monitor":"val_loss"}}

print("TFT v2 모델 정의 완료")

In [ ]:
# ===== 데이터셋 유틸 =====
def make_samples(full_df, start, end, seq_len, feats):
    samples=[]; s=pd.Timestamp(start); e=pd.Timestamp(end) if end else full_df["date"].max()
    for _,g in full_df.groupby("ticker"):
        g=g.sort_values("time_idx"); v=g[feats].values.astype(np.float32)
        t=g["target_5d"].values.astype(np.int64); d=g["date"].values
        for i in range(seq_len,len(g)):
            if d[i]>=s and d[i]<=e: samples.append((v[i-seq_len:i],t[i]))
    return samples

class SeqDS(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): x,y=self.s[i]; return torch.tensor(x),torch.tensor(y)

def predict_all(model, loader):
    ps,ls=[],[]
    model.eval(); model.cuda()
    with torch.no_grad():
        for x,y in loader:
            ps.extend(torch.softmax(model(x.cuda()),dim=-1)[:,1].cpu().numpy()); ls.extend(y.numpy())
    return np.array(ps),np.array(ls)

print("유틸 정의 완료")

In [ ]:
# ===== Walk-Forward 실행 (체크포인트 저장/로드 지원) =====
CKPT_DIR = MODEL_SAVE_PATH / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

wf_results = []

for i, (train_end, test_start, test_end) in enumerate(windows):
    print(f"\n{'='*60}")
    print(f"  Window [{i:2d}/{len(windows)}] train_end={train_end}, test={test_start}~{test_end}")
    print(f"{'='*60}")
    
    ckpt_path = CKPT_DIR / f"window_{i:02d}_te_{train_end}.ckpt"
    
    try:
        # 테스트 데이터 생성
        test_samples = make_samples(df, test_start, test_end, MAX_ENCODER_LENGTH, CLEANED_FEATURES)
        
        if len(test_samples) < 100:
            print(f"  SKIP: test={len(test_samples)} (데이터 부족)")
            wf_results.append({"window": i, "train_end": train_end, "test_start": test_start,
                               "test_end": test_end, "error": "데이터 부족"})
            continue
        
        test_loader = DataLoader(SeqDS(test_samples), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)
        
        # 체크포인트 로드 or 신규 학습
        if ckpt_path.exists():
            print(f"  체크포인트 로드: {ckpt_path.name}")
            best = TFTv2.load_from_checkpoint(str(ckpt_path), strict=False)
        else:
            print(f"  신규 학습...")
            train_samples = make_samples(df, "2008-01-01", train_end, MAX_ENCODER_LENGTH, CLEANED_FEATURES)
            
            if len(train_samples) < 1000:
                print(f"  SKIP: train={len(train_samples)} (데이터 부족)")
                wf_results.append({"window": i, "train_end": train_end, "test_start": test_start,
                                   "test_end": test_end, "error": "데이터 부족"})
                continue
            
            # Class weight
            labels = [s[1] for s in train_samples]
            n0 = sum(1 for l in labels if l==0); n1 = sum(1 for l in labels if l==1)
            cw = torch.tensor([len(labels)/(2*max(n0,1)), len(labels)/(2*max(n1,1))], dtype=torch.float32)
            
            pl.seed_everything(42)
            model = TFTv2(N_FEATURES, MAX_ENCODER_LENGTH, HIDDEN_SIZE, ATTENTION_HEADS, 1, DROPOUT, 2, LEARNING_RATE, cw)
            
            es = EarlyStopping(monitor="val_loss", patience=5, mode="min", verbose=False)
            trainer = pl.Trainer(max_epochs=MAX_EPOCHS, accelerator="gpu", devices=1,
                gradient_clip_val=0.1, callbacks=[es], enable_progress_bar=False,
                enable_model_summary=False, log_every_n_steps=100)
            
            val_size = max(len(train_samples)//10, 100)
            trainer.fit(model,
                train_dataloaders=DataLoader(SeqDS(train_samples[:-val_size]), batch_size=BATCH_SIZE, shuffle=True, num_workers=0),
                val_dataloaders=DataLoader(SeqDS(train_samples[-val_size:]), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0))
            
            best = TFTv2.load_from_checkpoint(trainer.checkpoint_callback.best_model_path, strict=False)
            
            # 체크포인트 저장
            trainer.save_checkpoint(str(ckpt_path))
            print(f"  체크포인트 저장: {ckpt_path.name}")
            
            del model, trainer
        
        # 테스트 예측
        probs, actuals = predict_all(best, test_loader)
        preds = (probs >= 0.5).astype(int)
        
        da = accuracy_score(actuals, preds)
        try: auc = roc_auc_score(actuals, probs)
        except: auc = float("nan")
        pos_rate = preds.mean()
        
        result = {"window": i, "train_end": train_end, "test_start": test_start,
                  "test_end": test_end, "n_test": len(test_samples),
                  "da": round(da*100, 2), "auc": round(auc, 4), "pos_rate": round(pos_rate*100, 1),
                  "ckpt": str(ckpt_path)}
        wf_results.append(result)
        print(f"  DA={da*100:.2f}%, AUC={auc:.4f}, 양성={pos_rate*100:.1f}%")
        
        del best
        torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"  ERROR: {e}")
        wf_results.append({"window": i, "train_end": train_end, "test_start": test_start,
                           "test_end": test_end, "error": str(e)})
        torch.cuda.empty_cache()

print(f"\nWalk-Forward 완료: {len(wf_results)}개 윈도우")
print(f"체크포인트 저장 위치: {CKPT_DIR}")

In [ ]:
# ===== 결과 요약 =====
valid = [r for r in wf_results if "error" not in r]

if valid:
    das = [r["da"] for r in valid]
    aucs = [r["auc"] for r in valid if not np.isnan(r["auc"])]
    
    print("="*70)
    print("  TFT v2 Walk-Forward 결과")
    print("="*70)
    print(f"  윈도우: {len(valid)}개 성공 / {len(wf_results)}개 전체")
    print(f"  평균 DA: {np.mean(das):.2f}% (std={np.std(das):.2f})")
    print(f"  평균 AUC: {np.mean(aucs):.4f}")
    print(f"  DA 범위: {min(das):.2f}% ~ {max(das):.2f}%")
    print()
    print(f"  {'윈도우':>6} {'기간':>24} {'DA':>8} {'AUC':>8} {'양성':>6}")
    print(f"  {'-'*60}")
    for r in valid:
        print(f"  [{r['window']:2d}]  {r['test_start']}~{r['test_end']}  {r['da']:6.2f}%  {r['auc']:.4f}  {r['pos_rate']:5.1f}%")
    print("="*70)
    print(f"\n비교: LSTM WF 평균 DA=48.9%, LightGBM WF 평균 DA=50.8%")
else:
    print("유효한 결과 없음")

In [ ]:
# ===== 저장 =====
json.dump(wf_results, open(str(MODEL_SAVE_PATH / f"wf_results_{datetime.now().strftime('%Y%m%d')}.json"), "w"),
          ensure_ascii=False, indent=2)
print(f"저장: {MODEL_SAVE_PATH}")

## TFT v2 Walk-Forward 검증\n\n- 3개월 단위 슬라이딩 윈도우 (2021-01 ~ 2026-01)\n- 각 윈도우: 과거 전체로 학습 -> 다음 3개월 예측\n- DA, AUC 시간 안정성 확인\n\n비교 기준: LSTM DA=48.9%, LightGBM DA=50.8%